# 原 39 项输入的训练期漂移与 f696 区间对照

冻结只读诊断。原始特征与账户均来自封存 Qlib Recorder，市场日历从同一 Qlib provider 取得；没有拟合或策略变更。

In [1]:
import os
os.environ["MLFLOW_ALLOW_FILE_STORE"] = "true"
import hashlib, json
from pathlib import Path
import numpy as np
import pandas as pd
import qlib
from qlib.data import D
from qlib.workflow import R

root = Path.cwd()
card = root / "research/stage1_f696_original39_input_drift_20261024"
plan_path = card / "plan.json"
plan = json.loads(plan_path.read_text())
assert plan["frozen"] and all(v == 0 for v in plan["budget"].values())
src = plan["sources"]
sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
feature_card = root / src["features_card"]
account_card = root / src["account_card"]
assert sha(feature_card / "summary.json") == src["features_summary_sha256"]
assert sha(feature_card / "nas_readback.json") == src["features_nas_readback_sha256"]
assert sha(account_card / "summary.json") == src["account_summary_sha256"]
feature_summary = json.loads((feature_card / "summary.json").read_text())
feature_nas = json.loads((feature_card / "nas_readback.json").read_text())
account_summary = json.loads((account_card / "summary.json").read_text())
assert feature_summary["artifact_policy"]["recorder_archive_sha256"] == src["features_nas_archive_sha256"]
assert feature_summary["nas_evidence"]["sha256"] == src["features_nas_archive_sha256"] and feature_summary["nas_evidence"]["size_bytes"] == src["features_nas_size_bytes"]
assert feature_nas["NAS_archive_sha256"] == src["features_nas_archive_sha256"] and feature_nas["artifact_hashes_checked"] >= 118
assert account_summary["nas_readback"]["verified"] and account_summary["nas_readback"]["sha256"] == src["account_nas_sha256"]
assert account_summary["nas_evidence"]["sha256"] == src["account_nas_sha256"] and account_summary["nas_evidence"]["size_bytes"] == src["account_nas_size_bytes"]
feature_artifacts = feature_card / "qlib_recorder" / src["features_experiment_id"] / src["features_recorder_id"] / "artifacts"
account_artifacts = account_card / "qlib_recorder" / src["account_experiment_id"] / src["account_recorder_id"] / "artifacts"
assert sha(feature_artifacts / "inputs/features") == src["features_artifact_sha256"]
assert sha(feature_artifacts / "inputs/timing") == src["timing_artifact_sha256"]
assert sha(account_artifacts / "analysis/daily") == src["account_daily_sha256"]
assert sha(account_artifacts / "analysis/rolling126") == src["account_rolling126_sha256"]
assert sha(root / src["provider"] / "qlib_store_manifest.json") == src["provider_manifest_sha256"]
qlib.init(provider_uri=str((root / src["provider"]).resolve()), region="us", expression_cache=None, dataset_cache=None)
with R.uri_context(str(feature_card / "qlib_recorder")):
    rec = R.get_recorder(experiment_id=src["features_experiment_id"], recorder_id=src["features_recorder_id"])
    features = rec.load_object("inputs/features")
    timing = rec.load_object("inputs/timing")
with R.uri_context(str(account_card / "qlib_recorder")):
    rec = R.get_recorder(experiment_id=src["account_experiment_id"], recorder_id=src["account_recorder_id"])
    daily = rec.load_object("analysis/daily").sort_values("trade_date").reset_index(drop=True)
    rolling = rec.load_object("analysis/rolling126")
original = feature_summary["input_audit"]["original_features"]
assert len(original) == 39 and list(features.columns[:39]) == original
assert len(features) == len(timing) == 3966 and len(daily) == 944 and len(rolling) == 819
calendar = pd.DatetimeIndex(D.calendar(start_time="2011-01-03", end_time="2026-10-09", freq="day"))
assert features.index.equals(calendar) and timing.index.equals(calendar)
held_dates = pd.DatetimeIndex(daily.signal_date)
assert held_dates.is_unique and held_dates.isin(features.index).all()
decisions = pd.to_datetime(timing.loc[held_dates, "decision_at"], utc=True).dt.tz_convert("America/New_York").dt.date
assert all(x > y.date() for x, y in zip(decisions, held_dates))
train = features.loc["2011-01-03":"2022-12-30", original]
held = features.loc[held_dates, original]
assert len(train) == 3020 and len(held) == 944
med = train.median(skipna=True)
scale = train.std(skipna=True, ddof=0)
assert np.isfinite(med.to_numpy()).all() and np.isfinite(scale.to_numpy()).all() and (scale > 0).all()
q01 = train.quantile(.01)
q99 = train.quantile(.99)
z = held.sub(med).div(scale)
families = {name: original[start:end+1] for name, (start,end) in plan["fixed_scope"]["families_by_original_feature_index"].items()}
family = pd.DataFrame(index=held.index)
for name, columns in families.items():
    valid = z[columns].notna().all(axis=1)
    family[name] = z[columns].abs().mean(axis=1).where(valid)
assert family.notna().all().all()
annual = []
feature_years = []
for year in (2023, 2024, 2025, 2026):
    dates = held.index[held.index.year == year]
    h = held.loc[dates]
    entry = {"year": year, "sessions": len(h), "partial_endpoint": year in (2023,2026),
             "family_mean_abs_train_z": {name: float(family.loc[dates,name].mean()) for name in families}}
    annual.append(entry)
    for name in original:
        hfinite = h[name].dropna()
        feature_years.append({"year": year, "feature": name, "train_finite": int(train[name].notna().sum()),
                              "held_finite": len(hfinite), "held_sessions": len(h),
                              "median_shift_train_sd": float((hfinite.median()-med[name])/scale[name]),
                              "outside_train_1_99_fraction": float(((hfinite<q01[name]) | (hfinite>q99[name])).mean())})
assert sum(a["sessions"] for a in annual) == 944 and len(feature_years) == 156
close = D.features(["SOXX"], ["$close"], start_time="2011-01-03", end_time="2026-10-09", freq="day")["$close"].droplevel("instrument").sort_index()
assert close.index.equals(calendar) and close.notna().all()
windows = []
for i in range(819):
    rw = rolling.iloc[i]
    segment = daily.iloc[i:i+126]
    assert pd.Timestamp(segment.trade_date.iloc[0]) == pd.Timestamp(rw.first_trade)
    assert abs(float(segment.paired_log_excess.sum()) - float(rw.relative_log_excess)) < 1e-10
    signal = pd.Timestamp(segment.signal_date.iloc[0])
    loc = calendar.get_loc(signal)
    assert loc >= 20
    r5 = float(close.iloc[loc]/close.iloc[loc-5]-1)
    r20 = float(close.iloc[loc]/close.iloc[loc-20]-1)
    state = "uptrend" if r5>=0 and r20>=0 else "rebound" if r5>=0 else "pullback" if r20>=0 else "decline"
    row = {"year": int(pd.Timestamp(rw.first_trade).year), "start_state": state,
           "outcome": "weak" if rw.relative_log_excess<0 else "good" if rw.relative_log_excess>0 else "tie"}
    for name in families:
        vals = family.loc[pd.DatetimeIndex(segment.signal_date),name]
        assert len(vals)==126 and vals.notna().all()
        row[name] = float(vals.mean())
    windows.append(row)
win = pd.DataFrame(windows)
assert win.outcome.value_counts().to_dict() == {"weak":564,"good":255}
cells = []
for (state,year),g in win.groupby(["start_state","year"],sort=True):
    a=g.loc[g.outcome.eq("weak")]; b=g.loc[g.outcome.eq("good")]
    c={"start_state":state,"year":int(year),"weak_windows":len(a),"good_windows":len(b),"matched":bool(len(a) and len(b))}
    if c["matched"]:
        c["weak_minus_good_family_magnitude"]={name:float(a[name].mean()-b[name].mean()) for name in families}
    cells.append(c)
matched=[c for c in cells if c["matched"]]
assert len(matched)==12 and all(c["good_windows"]==0 for c in cells if c["year"]==2023)
weights=np.array([min(c["weak_windows"],c["good_windows"]) for c in matched],dtype=float)
weighted={name:float(np.average([c["weak_minus_good_family_magnitude"][name] for c in matched],weights=weights)) for name in families}
signs={name:{"positive_cells":sum(c["weak_minus_good_family_magnitude"][name]>0 for c in matched),
             "negative_cells":sum(c["weak_minus_good_family_magnitude"][name]<0 for c in matched)} for name in families}
result={"status":"read_only_original39_input_drift_descriptive_no_factor_selected",
        "plan_sha256":sha(plan_path),"features_recorder_id":src["features_recorder_id"],"account_recorder_id":src["account_recorder_id"],
        "source_archive_sha256":src["features_nas_archive_sha256"],"account_archive_sha256":src["account_nas_sha256"],
        "train_sessions":len(train),"held_sessions":len(held),"rolling126_windows":len(win),
        "families":{k:list(v) for k,v in families.items()},"annual":annual,"feature_years":feature_years,
        "matched_state_year_cells":len(matched),"total_state_year_cells":len(cells),
        "matched_weak_minus_good_family_magnitude":weighted,"matched_signs":signs,"cells":cells,
        "new_market_requests":0,"new_model_fits":0,"new_native_accounts":0,"new_bootstraps":0,"new_recorders":0,
        "factor_selected":False,"policy_inference":False,"independent_validation":False,
        "limits":["All 39 feature drift statistics are descriptive and not ranked for selection",
                  "Weak/good groups use future returns and 126-session windows overlap",
                  "Four 2023 state-year cells have no good window for matched comparison",
                  "Feature distribution shift does not prove causal importance or incremental predictive value"]}
(card/"result.json").write_text(json.dumps(result,ensure_ascii=False,indent=2)+"\n")
print({"train":len(train),"held":len(held),"annual":annual,"matched":weighted,"signs":signs})


2026/10/24 16:10:17 INFO mlflow.agent.hint: Load the `instrumenting-with-mlflow-tracing` skill at /Users/rziiiii/dev/workspace/qlib_venv/lib/python3.12/site-packages/mlflow/assistant/skills/instrumenting-with-mlflow-tracing/SKILL.md before writing any tracing code; it ships with this MLflow install. Set MLFLOW_DISABLE_AGENT_HINT=1 to silence this.


[56515:MainThread](2026-10-24 16:10:17,250) INFO - qlib.Initialization - [config.py:452] - default_conf: client.


[56515:MainThread](2026-10-24 16:10:17,251) INFO - qlib.Initialization - [__init__.py:75] - qlib successfully initialized based on client settings.


[56515:MainThread](2026-10-24 16:10:17,251) INFO - qlib.Initialization - [__init__.py:77] - data_path={'__DEFAULT_FREQ': PosixPath('/Users/rziiiii/dev/workspace/data/qlib/minute_information_pit_allocator_20261012/daily_prices')}


{'train': 3020, 'held': 944, 'annual': [{'year': 2023, 'sessions': 250, 'partial_endpoint': True, 'family_mean_abs_train_z': {'trend_relative_breadth': 0.8316683785482455, 'volatility_risk': 0.5886397102163983, 'tracking_structure': 0.6970350368371535}}, {'year': 2024, 'sessions': 252, 'partial_endpoint': False, 'family_mean_abs_train_z': {'trend_relative_breadth': 0.8570452818467704, 'volatility_risk': 0.765188098741659, 'tracking_structure': 0.7724616247643722}}, {'year': 2025, 'sessions': 250, 'partial_endpoint': False, 'family_mean_abs_train_z': {'trend_relative_breadth': 1.023609565738443, 'volatility_risk': 0.9710537283962272, 'tracking_structure': 0.783327451777498}}, {'year': 2026, 'sessions': 192, 'partial_endpoint': True, 'family_mean_abs_train_z': {'trend_relative_breadth': 1.606547250456219, 'volatility_risk': 1.1515839696476815, 'tracking_structure': 1.1387255768113513}}], 'matched': {'trend_relative_breadth': -0.104558145044386, 'volatility_risk': 0.03231978575651933, 'tr

## 中文结论

按冻结计划只读 Qlib 原 39 项输入 Recorder、Qlib 日历和精确 f696 A0 0bp Tiered 账户；逐件核对 SHA、NAS 回读、信号日早于决策日、3020 个训练日、944 个评价日和全部 819 个连续 126 日窗口。无新行情请求、拟合、账户或 bootstrap。

以 2011–2022 训练期的中位数和标准差固定标准化，2026 年三组输入的平均绝对偏离分别为趋势／相对／广度 1.607、波动风险 1.152、跟踪结构 1.139；2023 年分别为 0.832、0.589、0.697。按训练期第 1/99 百分位定义，2026 年 39 项中有 20 项在超过 10% 的评价日越界，2023 年为 0 项。全部 39 项按年报告在 `result.json`，未按评价期收益筛选或排序。

输入漂移**不足以单独解释**账户坏区间：2023 年的四个“事前走势 × 年份”格子全部没有好窗口，而其越界比例并不极端。在其余 12 个可配对格子中，弱减好的家族绝对标准化偏离，按每格较小组窗口数加权分别为趋势／相对／广度 -0.105、波动风险 +0.032、跟踪结构 -0.097；同格正负数量分别为 6/6、7/5、3/9。窗口重叠且好坏由未来收益定义，不能将这些差值视为独立统计检验、因果特征重要性或开关阈值。

本固定诊断决定为 `descriptive_2026_input_shift_not_sufficient_to_explain_exact_f696_weak_intervals`。不能仅凭 2026 输入漂移对原因子做截断、缩放或按年份重校准。下一步要在保存的原模型输出中区分：原 39 项已含的信息是否没有被正确接收，还是输出形成配置意图后又被现金配置和自身风险容量共同改变。任何新因子需有新的事前机制、匹配接收任务及完整 0bp 账户；SOXX v2 未通过。
